# App Reflexes — results notebook

Runs the full pipeline and displays the comparison table, equity curve, and null test.
**Read `../HONEST_ASSESSMENT.md` before trusting any number here.**

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from reflexes.config import load_config
from reflexes.data.ingest import load_universe, to_panel
from reflexes.features.assemble import build_feature_panel
from reflexes.decision.base import make_combiner
from reflexes.portfolio.risk import apply_risk_limits
from reflexes.backtest.engine import run_backtest
from reflexes.backtest.metrics import compute_metrics, metrics_table
from reflexes.backtest.benchmarks import buy_and_hold_weights, equal_weight_weights
from reflexes.backtest.walkforward import split_is_oos
from reflexes.backtest.null_test import run_null_test, null_percentile

cfg = load_config(ROOT / 'config' / 'default.yaml')
cfg['data']['cache_dir'] = str(ROOT / 'data_cache')
cfg['decision']['combiner']

In [ ]:
# 1. Data -> 2. Features -> 3. Decision -> 4. Risk
frames = load_universe(cfg)
asset_returns = to_panel(frames, 'close').pct_change().fillna(0.0)
signals = build_feature_panel(frames, cfg)

combiner = make_combiner(cfg)
common = None
for sf in signals.values():
    idx = sf.dropna(how='any').index
    common = idx if common is None else common.intersection(idx)
raw = combiner.run(signals, dates=common).fillna(0.0)
risked = apply_risk_limits(raw, asset_returns, cfg)
risked.tail()

In [ ]:
# 5. Backtest strategy + benchmarks (same engine, same costs)
strat = run_backtest(frames, risked, cfg, verify=True)
assets, idx = list(risked.columns), risked.index
bh = run_backtest(frames, buy_and_hold_weights(idx, assets, 'BTC'), cfg)
ew = run_backtest(frames, equal_weight_weights(idx, assets), cfg)
oos_start = cfg['backtest']['oos_start']

def is_oos(res):
    ir, orr = split_is_oos(res.returns, oos_start)
    ie, oe = split_is_oos(res.equity, oos_start)
    it, ot = split_is_oos(res.turnover, oos_start)
    ic, oc = split_is_oos(res.costs, oos_start)
    return {'IS': compute_metrics(ir, ie, it, ic), 'OOS': compute_metrics(orr, oe, ot, oc)}

table = metrics_table({
    'STRAT IS': is_oos(strat)['IS'], 'STRAT OOS': is_oos(strat)['OOS'],
    'BTC B&H OOS': is_oos(bh)['OOS'], 'EqualWt OOS': is_oos(ew)['OOS'],
})
table.loc[['cagr','sharpe','sortino','max_drawdown','hit_rate','turnover_annual','total_costs']].round(3)

In [ ]:
# Equity curves (net of costs), log scale
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(strat.equity, label='Strategy'); ax.plot(bh.equity, label='BTC B&H'); ax.plot(ew.equity, label='Equal weight')
ax.axvline(pd.Timestamp(oos_start, tz='UTC'), color='k', ls='--', alpha=0.6)
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3); ax.set_title('Strategy vs benchmarks (net of costs)')
plt.show()

In [ ]:
# 6. Null test: what does 'no edge' look like?
null = run_null_test(frames, raw, asset_returns, cfg, mode='shuffle')
strat_oos_sharpe = is_oos(strat)['OOS']['sharpe']
print('null OOS Sharpe  mean=%.3f  p95=%.3f' % (null['oos_sharpe_mean'], null['oos_sharpe_p95']))
print('strategy OOS Sharpe = %.3f  (beats %.0f%% of null runs)' % (
    strat_oos_sharpe, 100*null_percentile(null, strat_oos_sharpe)))
print('\nNOTE: the null is dominated by cost drag on churn, so beating it is weak evidence.\n'
      'The real test is vs the benchmarks above — see HONEST_ASSESSMENT.md.')